<a href="https://colab.research.google.com/github/Mitochondrion3000/epic-competiton/blob/claude/gracious-brown-4ybsmj/simple_model/step3b_positional.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Простая интерпретируемая модель. Шаг 3б: позиционные признаки и честная (плотная) валидация

На шаге 3 сайт каждого фактора считался в одном общем окне −150..+50. Но на профилях видно, что у каждого фактора своё
место: YY1 сидит прямо на старте, NF-Y на −110..−40, HINFP чуть ниже старта. Здесь:
1. **Плотная валидация.** До сих пор отрицательные позиции брались случайно по геному, а в тесте оцениваются *все* позиции
   подряд, в том числе соседи стартов (это примерно 10% whitelist). Берём половину валидационной хромосомы целиком
   (~13.5 млн позиций-цепей) и считаем метрики **ровно как официальный скорер**.
2. **Окно пика для каждого фактора**: выбираем его по профилю обогащения.
3. **Локальный контраст Initiator**: насколько эта позиция лучше соседних по мотиву старта.
4. **Сравниваем варианты и сохраняем финальные модели.**
5. **Подбираем квантование сабмита** (показатель `alpha`), которое нужно для шага 4.

## 0. Настройки и данные

In [ ]:
import os, sys, gzip, glob, json, hashlib, tarfile, time, re, urllib.request
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
try:
    import lightgbm as lgb
except ImportError:
    os.system("pip install -q lightgbm"); import lightgbm as lgb

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

SPECIES = os.environ.get("EPIC_SPECIES", "jaNemVect1.1")
ON_COLAB = "google.colab" in sys.modules
if ON_COLAB:
    from google.colab import drive; drive.mount("/content/drive")
DATA_DIR = os.environ.get("EPIC_DATA_DIR", "/content/epic")
OUT_DIR = os.environ.get("EPIC_OUT_DIR", "/content/drive/MyDrive/epic_simple" if ON_COLAB else "/content/epic_simple")
BASE, FP = f"{DATA_DIR}/{SPECIES}", f"{DATA_DIR}/{SPECIES}/for_participants"

RAW = "https://raw.githubusercontent.com/Mitochondrion3000/epic-competiton/claude/gracious-brown-4ybsmj/simple_model/"
for mod in ("features.py", "motifs.py", "scoring.py"):
    if not os.path.exists(mod): urllib.request.urlretrieve(RAW + mod, mod)
import features as F, motifs as M, scoring as S

table = pd.read_parquet(f"{OUT_DIR}/{SPECIES}.step1_table.parquet")
meta = json.load(open(f"{OUT_DIR}/{SPECIES}.step1_meta.json"))
p3 = json.load(open(f"{OUT_DIR}/{SPECIES}.step3_params.json"))
print(f"таблица: {len(table):,} строк; признаков шага 3: {len(p3['features'])}")

In [ ]:
def download_dataset(species, data_dir):
    if os.path.isdir(f"{data_dir}/{species}"): return
    import requests
    os.makedirs(data_dir, exist_ok=True)
    rec = requests.get("https://zenodo.org/api/records/22285753", timeout=60).json()
    f = [x for x in rec["files"] if x["key"] == f"{species}.tgz"][0]
    dst, md5 = f"{data_dir}/{species}.tgz", hashlib.md5()
    with requests.get(f["links"]["self"], stream=True, timeout=120) as r, open(dst, "wb") as fh:
        r.raise_for_status()
        for chunk in tqdm(r.iter_content(1 << 22), total=f["size"] // (1 << 22) + 1, desc=f"скачиваю {species}"):
            fh.write(chunk); md5.update(chunk)
    assert md5.hexdigest() == f.get("checksum", "md5:").split(":")[-1], "MD5 не совпал"
    with tarfile.open(dst) as tf:
        try: tf.extractall(data_dir, filter="data")
        except TypeError: tf.extractall(data_dir)
    os.remove(dst)

CODE = np.full(256, 4, np.uint8)
for i, b in enumerate("ACGT"):
    CODE[ord(b)] = CODE[ord(b.lower())] = i

def read_genome(path, keep):
    seq, low, name, chunks = {}, {}, None, []
    def flush():
        if name in keep:
            raw = np.frombuffer(b"".join(chunks), np.uint8)
            seq[name], low[name] = CODE[raw], raw >= ord("a")
    with gzip.open(path, "rb") as fh:
        for line in fh:
            if line.startswith(b">"):
                flush(); name, chunks = line[1:].split()[0].decode(), []
            elif name in keep:
                chunks.append(line.rstrip())
    flush()
    return seq, low

download_dataset(SPECIES, DATA_DIR)
SEQ, LOW = read_genome(f"{BASE}/genome/genome.fa.gz", set(table.chrom))
wl_train = pd.read_csv(f"{FP}/whitelist.train.bed.gz", sep="\t", header=None, usecols=[0, 1, 2], names=["chrom", "start", "end"], dtype={"chrom": str})

## 1. Плотная валидация

Берём первую половину первого валидационного контига и **все** позиции whitelist в ней на обеих цепях. Метки берём из
таблицы шага 1 (там все старты). Метрики считаем функциями `scoring.py`, это копия официального скорера:
* **AUPRC** по всем позициям;
* **Spearman** = корреляция Пирсона между *плотными рангами* предсказания и правды среди стартов. Половина стартов
  имеет счёт 1, у них одинаковый ранг, поэтому это число заметно отличается от обычного Spearman, который мы считали
  на шаге 3 (0.32). Дальше везде используем официальную формулу.

In [ ]:
DENSE_CONTIG = meta["VAL_CONTIGS"][0]
L = len(SEQ[DENSE_CONTIG]); inwl = np.zeros(L, bool)
for s, e in zip(*wl_train[wl_train.chrom == DENSE_CONTIG][["start", "end"]].values.T): inwl[s:e] = True
inwl[L // 2:] = False
dpos = np.flatnonzero(inwl)
D_POS = np.concatenate([dpos, dpos]); D_STRAND = np.array(["+"] * len(dpos) + ["-"] * len(dpos))
lab = table[(table.chrom == DENSE_CONTIG) & (table.y == 1)]
tot = dict(zip(lab.pos.values * 2 + (lab.strand.values == "-"), lab.tot.values))
D_Y = np.array([tot.get(k, 0.0) for k in (D_POS * 2 + (D_STRAND == "-")).tolist()], np.float32)
print(f"плотная валидация: {DENSE_CONTIG}, {len(D_Y):,} позиций-цепей, стартов {(D_Y > 0).sum():,}, доля {(D_Y > 0).mean():.2e}")

def dense_metrics(score):
    # метрики по «сырому» счёту, без квантования сабмита (оно будет в разделе 5)
    from sklearn.metrics import average_precision_score
    y = D_Y > 0
    return {"AUPRC": average_precision_score(y, score), "Spearman (офиц.)": S.spearman_dense(score[y], D_Y[y])}

## 2. Окно пика для каждого фактора

Для каждого из 12 факторов шага 3 строим профиль (как на шаге 3): во сколько раз сайт чаще у сильных стартов, чем у
случайных позиций, по смещению. Находим максимум в −150..+50 и расширяем окно влево и вправо, пока обогащение не
упадёт вдвое относительно пика. Это окно и становится новым признаком `tf_<фактор>_peak`.

In [ ]:
A, B = -200, 100
fit = table[table.split == "fit"]
def windows(df, a=A, b=B):
    off = np.arange(a, b + 1); out = np.full((len(df), len(off)), 4, np.uint8)
    for (c, st), d in df.groupby(["chrom", "strand"]):
        s = SEQ[c]; rows = df.index.get_indexer(d.index)
        g = d.pos.values[:, None] + (off if st == "+" else -off)[None, :]
        ok = (g >= 0) & (g < len(s)); gg = np.where(ok, g, 0)
        w = s[gg] if st == "+" else F.COMP[s[gg]]
        out[rows] = np.where(ok, w, 4)
    return out
XS = M.onehot(windows(fit[fit.tot >= 10].sample(6000, random_state=1).reset_index(drop=True)))
XN = M.onehot(windows(fit[fit.y == 0].sample(10000, random_state=1).reset_index(drop=True)))
smooth = lambda x, k=15: np.convolve(x, np.ones(k) / k, mode="same")

TF = [dict(m) for m in p3["tf"]]
ncol = 4; nrow = int(np.ceil(len(TF) / ncol))
fig, axs = plt.subplots(nrow, ncol, figsize=(17, 2.8 * nrow), sharex=True)
rows = []
for ax, m in zip(axs.ravel(), TF):
    pw = np.array(m["pwm"], np.float32)[None]; L1 = XS.shape[1] - pw.shape[1] + 1
    fs = smooth((M.window_scores(XS, pw, 0, L1)[..., 0] >= m["thr"]).mean(0))
    fn = smooth((M.window_scores(XN, pw, 0, L1)[..., 0] >= m["thr"]).mean(0))
    lr = np.log2((fs + 1e-4) / (fn + 1e-4)); off = np.arange(A, A + L1)
    k = int(np.argmax(np.where((off >= -150) & (off <= 50), lr, -9))); a = b = k
    while a > 0 and lr[a - 1] >= lr[k] - 1: a -= 1
    while b < L1 - 1 and lr[b + 1] >= lr[k] - 1: b += 1
    m["windows"] = {**{kk: tuple(v) for kk, v in m["windows"].items()}, "_peak": (int(off[a]), int(off[b]))}
    rows.append({"фактор": m["name"], "окно пика": f"{off[a]:+d} .. {off[b]:+d}", "обогащение в пике, раз": 2 ** lr[k]})
    ax.plot(off, lr, color="darkred", lw=1.2); ax.axhline(0, color="k", lw=.6); ax.axvline(0, color="gray", ls=":", lw=.8)
    ax.axvspan(-150, 50, color="tab:green", alpha=.07, label="старое окно")
    ax.axvspan(off[a], off[b], color="tab:orange", alpha=.35, label="окно пика")
    ax.set_title(m["name"], fontsize=9); ax.set_ylim(-2, 6)
for ax in axs.ravel()[len(TF):]: ax.axis("off")
axs[0, 0].legend(fontsize=7)
plt.suptitle("Окно пика каждого фактора (оранжевое) против общего окна шага 3 (зелёное); log2(сильные старты / случайные)", y=1.0)
plt.tight_layout(); plt.show()
pd.DataFrame(rows).style.format({"обогащение в пике, раз": "{:.1f}"})

COMMENT_PEAKS

## 3. Признаки: окна пиков и локальный контраст Initiator

* `tf_<фактор>_peak`: число сайтов фактора в его окне пика.
* `inr_vs_3`, `inr_vs_10`, `inr_vs_25`: `inr_pwm` этой позиции минус лучший `inr_pwm` среди соседей в ±3, ±10, ±25 нт.
  0 означает «это лучшая позиция по мотиву старта в округе», отрицательное значение — «рядом есть более похожая».
  Идея: внутри промотора старт выбирает конкретную букву, и это должна быть локально лучшая позиция.

Считаем всё для таблицы (~25 мин) и для плотной валидации (~3 мин).

In [ ]:
PARAMS = {"pwm": np.array(p3["pwm"]), "kmers": [tuple(k) for k in p3["kmers"]], "tf": TF, "wide": True, "local": True}
t0 = time.time(); parts = []
for c, d in tqdm(table.groupby("chrom", sort=False), desc="контиги", total=table.chrom.nunique()):
    parts.append(pd.DataFrame(F.contig_features(SEQ[c], LOW[c], d.pos.values, d.strand.values, PARAMS), index=d.index))
feat = pd.concat(parts).loc[table.index]; del parts
FEATURES = list(feat.columns)
data = pd.concat([table, feat], axis=1); del feat
print(f"таблица: {len(FEATURES)} признаков за {(time.time() - t0) / 60:.1f} мин")
t0 = time.time()
DX = F.feature_matrix(SEQ[DENSE_CONTIG], LOW[DENSE_CONTIG], D_POS, D_STRAND, PARAMS, FEATURES)
print(f"плотная валидация: матрица {DX.shape} за {(time.time() - t0) / 60:.1f} мин")
json.dump({"pwm": p3["pwm"], "kmers": p3["kmers"], "tf": TF, "wide": True, "local": True, "features": FEATURES},
          open(f"{OUT_DIR}/{SPECIES}.step3b_params.json", "w"))

## 4. Что дают новые признаки (плотная валидация)

Учим те же модели, что на шаге 3 (аддитивную и деревья глубины 3), на `fit` с разными наборами признаков и считаем
метрики на плотной валидации. Итоговый счёт = `log P(старт) + E[log(1 + счёт) | старт]`, как на шаге 3.

In [ ]:
OLD = p3["features"]; LOCAL = [f for f in FEATURES if f.startswith("inr_vs")]; PEAK = [f for f in FEATURES if f.endswith("_peak")]
fit = data[data.split == "fit"]; pos_fit = fit[fit.y == 1]

def train(feats, depth):
    kw = dict(max_depth=depth, num_leaves=2 ** depth, learning_rate=0.05, n_estimators=800 if depth == 1 else 500,
              min_child_samples=200, subsample=0.8, subsample_freq=1, colsample_bytree=0.8, verbose=-1, random_state=0)
    clf = lgb.LGBMClassifier(**kw).fit(fit[feats], fit.y)
    reg = lgb.LGBMRegressor(**kw).fit(pos_fit[feats], np.log1p(pos_fit.tot))
    return clf, reg

def dense_score(clf, reg, feats):
    cols = [FEATURES.index(f) for f in feats]; X = DX[:, cols]
    return np.log(np.clip(clf.predict_proba(X)[:, 1], 1e-12, 1)) + reg.predict(X)

res, MODELS = {}, {}
for depth, mname in ((1, "аддитивная"), (3, "деревья глубины 3")):
    for vname, feats in (("признаки шага 3", OLD), ("+ локальный контраст", OLD + LOCAL), ("+ окна пиков", OLD + PEAK), ("+ всё", FEATURES)):
        clf, reg = train(feats, depth)
        res[(mname, vname)] = dense_metrics(dense_score(clf, reg, feats))
        if vname == "+ всё": MODELS[mname] = (clf, reg)
        print(mname, "|", vname, "|", {k: round(v, 4) for k, v in res[(mname, vname)].items()}, flush=True)
res = pd.DataFrame(res).T
res["AUPRC / случайный"] = res.AUPRC / (D_Y > 0).mean()
display(res.style.format("{:.4f}").format({"AUPRC / случайный": "{:.1f}"}).background_gradient(subset=["AUPRC"], cmap="Greens"))

COMMENT_GAIN

### Сколько добавляет один сайт фактора в «своём» окне

Аддитивная модель позволяет прочитать эффект каждого признака отдельно. Для признаков-окон пиков берём вклад в
log-шанс при 0 сайтов и при 1 сайте и переводим разницу в «во сколько раз выше шанс старта».

In [ ]:
clf_add = MODELS["аддитивная"][0]
sample = data[data.split == "val"].sample(200000, random_state=0)
contrib = clf_add.booster_.predict(sample[FEATURES], pred_contrib=True)[:, :-1]
eff = []
for f in PEAK + LOCAL + ["inr_pwm", "tf_TYY1"]:
    if f not in FEATURES: continue
    j = FEATURES.index(f); x = sample[f].values; c = contrib[:, j]
    if f.startswith("inr_vs") or f == "inr_pwm":
        lo, hi = np.quantile(x, [0.1, 0.9]); d = c[x >= hi].mean() - c[x <= lo].mean(); what = "верхние 10% против нижних 10%"
    else:
        if not (x >= 1).any(): continue
        d = c[x >= 1].mean() - c[x == 0].mean(); what = "≥ 1 сайт против 0"
    eff.append({"признак": f, "сравнение": what, "шанс выше, раз": np.exp(d), "доля позиций с сайтом": (x >= 1).mean() if "сайт" in what else np.nan})
eff = pd.DataFrame(eff).sort_values("шанс выше, раз", ascending=False)
display(eff.style.format({"шанс выше, раз": "{:.2f}", "доля позиций с сайтом": "{:.2%}"}))

COMMENT_EFFECTS

## 5. Квантование сабмита

Сабмит хранит 5 знаков после запятой, поэтому скорер видит не больше 100 001 разных значений. Как и в v2, переводим счёт
в `r^(-alpha)`, где `r` — место позиции в рейтинге (1 = лучшая). Порядок не меняется, но все позиции ниже некоторого
места округляются в одно значение и получают один плотный ранг. Это почти не трогает AUPRC (её определяют верхние
места), но помогает Spearman: слабые старты и так неразличимы для модели, и слияние их в один ранг уменьшает шум.
Перебираем `alpha` на плотной валидации и берём наибольший, при котором AUPRC не падает больше чем на 0.5% от лучшего.

In [ ]:
best_name = "деревья глубины 3"
clf, reg = MODELS[best_name]
raw = dense_score(clf, reg, FEATURES)
ALPHAS = (0.0, 0.3, 0.4, 0.5, 0.55, 0.6, 0.65, 0.7, 0.8)
grid = pd.DataFrame([{"alpha": a, **S.official_score(S.rank_power(raw, a), D_Y)} for a in ALPHAS]).set_index("alpha")
grid["AUPRC / лучший"] = grid.AUPRC / grid.AUPRC.max()
ALPHA = float(grid.index[grid["AUPRC / лучший"] >= 0.995].max())
display(grid.style.format("{:.4f}"))
print(f"выбран alpha = {ALPHA}: AUPRC {grid.loc[ALPHA, 'AUPRC']:.4f}, Spearman {grid.loc[ALPHA, 'Spearman']:.4f}")
fig, ax = plt.subplots(1, 2, figsize=(11, 3))
ax[0].plot(grid.index, grid.AUPRC, "o-"); ax[0].set(xlabel="alpha", title="AUPRC после квантования")
ax[1].plot(grid.index, grid.Spearman, "o-", color="tab:green"); ax[1].set(xlabel="alpha", title="Spearman после квантования")
for a in ax: a.axvline(ALPHA, color="tab:red", lw=1)
plt.tight_layout(); plt.show()

In [ ]:
# Сохраняем финальные модели для шага 4
FINAL = {"additive": MODELS["аддитивная"], "depth3": MODELS["деревья глубины 3"]}
for tag, (clf, reg) in FINAL.items():
    clf.booster_.save_model(f"{OUT_DIR}/{SPECIES}.step3b_{tag}_cls.txt"); reg.booster_.save_model(f"{OUT_DIR}/{SPECIES}.step3b_{tag}_reg.txt")
json.dump({"alpha": ALPHA, "dense_contig": DENSE_CONTIG, "dense_results": {f"{k[0]} | {k[1]}": v for k, v in res.drop(columns="AUPRC / случайный").to_dict("index").items()},
           "alpha_grid": grid.reset_index().to_dict("records")}, open(f"{OUT_DIR}/{SPECIES}.step3b_summary.json", "w"), default=float)
print("сохранено в", OUT_DIR)

## Итог шага 3б

COMMENT_SUMMARY